# Tool Use Pattern - Vehicle Sale Evidence


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f1a9fbda7-77a8-4a7a-ac2c-077fb98e53a6_716x552-1.gif" alt="General Tool Use Pattern" width="800"/>

This notebook uses the Tool Use Pattern to analyze a staged mobile-forensics case about vehicle photos and listing drafts. Instead of generic demo tools, you will work with the same local evidence package described in `02_case_overview.md`.

This is the second core lab in the course. If you want a refresher on iterative critique, review the [Reflection Pattern notebook](../lab1_reflection_pattern/03_lab_notebook.ipynb).

In this lab, the goal is not to trust tool output blindly. The goal is to choose valid tools, pass valid arguments, inspect the results, and produce a conclusion that stays inside the evidence.


## Lab Question and Report Format

Use the staged artifacts to answer this case question:

**Does the phone contain confirmed, likely, or unconfirmed evidence that a stolen black SUV was photographed and prepared for an online sale after January 2, 2026?**

Return the same five-part report format in both parts of the notebook:

1. `tool-call log`
2. `strongest timestamp evidence`
3. `strongest vehicle-match evidence`
4. `conclusion label (confirmed, likely, or unconfirmed) with confidence 0-1 per major claim`
5. `explicit evidence mapping and limits`

Required core tool sequence for the main case:
`list_media_files -> extract_image_metadata -> detect_vehicle_attributes -> inspect_listing_records -> compare_vehicle_features`


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and loads the case data.
LAB_NAME = 'lab2_tool_use_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f"MODEL or OLLAMA_BASE_URL is missing from {env_path}")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

data_dir = lab_dir / "data"
if not (data_dir / "artifact_manifest.json").exists():
    raise FileNotFoundError("Could not find lab2_tool_use_pattern/data")

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def load_csv_records(path: Path) -> list[dict]:
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

artifact_manifest = json.loads((data_dir / "artifact_manifest.json").read_text())
media_index = load_csv_records(data_dir / "media_index.csv")
image_metadata = load_csv_records(data_dir / "image_metadata.csv")
vehicle_detections = load_csv_records(data_dir / "vehicle_detections.csv")
listing_drafts = json.loads((data_dir / "listing_drafts.json").read_text())
chain_of_custody = load_csv_records(data_dir / "chain_of_custody.csv")

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


### Review the Staged Artifact Package

The lab works with local evidence only. Before you call any tools, inspect the case manifest so you know the incident window, acquisition details, and artifact set you are about to query.


In [ ]:
artifact_manifest


## Part 1: Run the Workflow Manually

In Part 1, you will define the forensic tools and run the required sequence directly. This makes the tool names, arguments, and raw outputs explicit before you hand the same tool set to `ToolAgent`.


In [ ]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.tool_pattern.tool_agent import ToolAgent

media_by_name = {row["file_name"]: row for row in media_index}
metadata_by_name = {row["file_name"]: row for row in image_metadata}
detections_by_name = {row["file_name"]: row for row in vehicle_detections}


def normalize_root(root: str) -> str:
    normalized = root.strip().strip("/")
    if normalized.lower() == "gallery":
        return "DCIM/Camera"
    return normalized


def as_json(payload) -> str:
    return json.dumps(payload, indent=2)


@tool
def list_media_files(root: str, date_from: str):
    """
    List media files from the requested folder on or after a given UTC timestamp.

    Args:
        root (str): Folder name such as "DCIM/Camera" or "gallery/".
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    normalized_root = normalize_root(root)
    matches = [
        {
            "file_name": row["file_name"],
            "folder": row["folder"],
            "created_at_utc": row["created_at_utc"],
            "path": row["path"],
            "size_bytes": row["size_bytes"],
        }
        for row in media_index
        if row["created_at_utc"] >= date_from and row["folder"].startswith(normalized_root)
    ]
    return as_json(matches)


@tool
def extract_image_metadata(file_name: str):
    """
    Extract metadata for one gallery image.

    Args:
        file_name (str): Image filename such as "IMG_2044.jpg".
    """
    record = metadata_by_name.get(file_name)
    if record is None:
        return as_json({"error": f"No metadata found for {file_name}"})
    return as_json(record)


@tool
def detect_vehicle_attributes(file_name: str):
    """
    Retrieve precomputed vehicle-detection results for one image.

    Args:
        file_name (str): Image filename such as "IMG_2044.jpg".
    """
    record = detections_by_name.get(file_name)
    if record is None:
        return as_json({"error": f"No detection found for {file_name}"})
    return as_json(record)


@tool
def inspect_listing_records(source: str, date_from: str):
    """
    Inspect saved listing-draft records created on or after a given UTC timestamp.

    Args:
        source (str): Listing source file name. Use "listing_drafts.json" for this lab.
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    if source != "listing_drafts.json":
        return as_json({"error": f"Unsupported source: {source}"})
    matches = [draft for draft in listing_drafts if draft["created_at_utc"] >= date_from]
    return as_json(matches)


@tool
def compare_vehicle_features(case_description: str, file_name: str):
    """
    Compare the case vehicle description with detected features for one image.

    Args:
        case_description (str): Known case description such as "black SUV with roof rack".
        file_name (str): Image filename such as "IMG_2044.jpg".
    """
    detection = detections_by_name.get(file_name)
    if detection is None:
        return as_json({"error": f"No detection found for {file_name}"})

    description = case_description.lower()
    matching = []
    conflicting = []

    if "black" in description:
        if detection["color"].lower() == "black":
            matching.append("black")
        else:
            conflicting.append(f"color={detection['color']}")
    if "suv" in description:
        if detection["body_type"].lower() == "suv":
            matching.append("SUV")
        else:
            conflicting.append(f"body_type={detection['body_type']}")
    if "roof rack" in description:
        if detection["roof_rack"].lower() == "yes":
            matching.append("roof rack")
        else:
            conflicting.append("roof rack not detected")

    if len(matching) == 3 and not conflicting:
        match_strength = "strong"
    elif len(matching) >= 2:
        match_strength = "moderate"
    else:
        match_strength = "weak"

    return as_json(
        {
            "file_name": file_name,
            "matching_features": matching,
            "conflicting_features": conflicting,
            "match_strength": match_strength,
            "detector_confidence": detection["confidence"],
        }
    )


forensic_tools = [
    list_media_files,
    extract_image_metadata,
    detect_vehicle_attributes,
    inspect_listing_records,
    compare_vehicle_features,
]


### Inspect the Tool Schemas

Before you run the case, inspect the generated tool signatures. This is the schema information the model will later use in `ToolAgent`.


In [ ]:
tool_catalog = {tool.name: json.loads(tool.fn_signature) for tool in forensic_tools}
tool_catalog


### Execute the Core Tool Sequence Manually

The next cell runs the required sequence directly against the staged case. Review the outputs before moving on to the report step.


In [ ]:
case_description = "black SUV with roof rack"
date_from = artifact_manifest["incident_window_utc"]["start"]

candidate_media = json.loads(list_media_files.run(root="DCIM/Camera", date_from=date_from))
img_2044_metadata = json.loads(extract_image_metadata.run(file_name="IMG_2044.jpg"))
img_2051_metadata = json.loads(extract_image_metadata.run(file_name="IMG_2051.jpg"))
img_2044_detection = json.loads(detect_vehicle_attributes.run(file_name="IMG_2044.jpg"))
img_2051_detection = json.loads(detect_vehicle_attributes.run(file_name="IMG_2051.jpg"))
listing_results = json.loads(inspect_listing_records.run(source="listing_drafts.json", date_from=date_from))
comparison_2044 = json.loads(compare_vehicle_features.run(case_description=case_description, file_name="IMG_2044.jpg"))
comparison_2051 = json.loads(compare_vehicle_features.run(case_description=case_description, file_name="IMG_2051.jpg"))

manual_tool_log = [
    {"tool": "list_media_files", "arguments": {"root": "DCIM/Camera", "date_from": date_from}},
    {"tool": "extract_image_metadata", "arguments": {"file_name": "IMG_2044.jpg"}},
    {"tool": "detect_vehicle_attributes", "arguments": {"file_name": "IMG_2044.jpg"}},
    {"tool": "inspect_listing_records", "arguments": {"source": "listing_drafts.json", "date_from": date_from}},
    {"tool": "compare_vehicle_features", "arguments": {"case_description": case_description, "file_name": "IMG_2044.jpg"}},
]

manual_evidence_bundle = {
    "candidate_media": candidate_media,
    "metadata_checks": {
        "IMG_2044.jpg": img_2044_metadata,
        "IMG_2051.jpg": img_2051_metadata,
    },
    "vehicle_checks": {
        "IMG_2044.jpg": img_2044_detection,
        "IMG_2051.jpg": img_2051_detection,
    },
    "listing_results": listing_results,
    "feature_comparisons": {
        "IMG_2044.jpg": comparison_2044,
        "IMG_2051.jpg": comparison_2051,
    },
}

manual_evidence_bundle


### Ask the Model to Write the Report from the Manual Outputs

The tool sequence above was executed directly by you. In this next step, the model only writes the report from the gathered evidence.


In [ ]:
REPORT_FORMAT = """
Return:
(1) tool-call log
(2) strongest timestamp evidence
(3) strongest vehicle-match evidence
(4) conclusion label (confirmed, likely, or unconfirmed) with confidence 0-1 per major claim
(5) explicit evidence mapping and limits
""".strip()

CASE_QUESTION = (
    "Does the phone contain confirmed, likely, or unconfirmed evidence that a stolen black SUV "
    "was photographed and prepared for an online sale after January 2, 2026?"
)

manual_prompt = f"""
You are a careful forensic analyst. Stay inside the evidence.
Use the manually collected tool outputs below to answer the case question.

Case question:
{CASE_QUESTION}

Tool-call log:
{json.dumps(manual_tool_log, indent=2)}

Tool outputs:
{json.dumps(manual_evidence_bundle, indent=2)}

{REPORT_FORMAT}
""".strip()


In [ ]:
manual_report = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a careful forensic analyst. Distinguish observed evidence from conclusions and state unresolved limits explicitly.",
        },
        {
            "role": "user",
            "content": manual_prompt,
        },
    ],
).choices[0].message.content

print(manual_report)


## Part 2: Run the Same Workflow with `ToolAgent`

In Part 2, you hand the same five forensic tools to `ToolAgent`. The report format stays the same, but the agent now decides which tools to call and when to stop.


In [ ]:
tool_agent = ToolAgent(tools=forensic_tools, client=client, model=MODEL)

agent_user_msg = f"""
Use the available forensic tools to answer this case question.

Case question:
{CASE_QUESTION}

Required core sequence:
list_media_files -> extract_image_metadata -> detect_vehicle_attributes -> inspect_listing_records -> compare_vehicle_features

Relevant dataset hints:
- the media folder is DCIM/Camera
- the saved listing records are in listing_drafts.json
- the known vehicle description is: {case_description}

Focus on the strongest image candidate, but mention competing evidence if it weakens the conclusion.

{REPORT_FORMAT}
""".strip()


In [ ]:
agent_report = tool_agent.run(user_msg=agent_user_msg)
print(agent_report)


## Optional Extension: Retrieval-Style Helper

The core lab stops here. If you want one glimpse beyond basic tool use, you can wrap a small search helper as a tool without introducing a full vector database. This is only a bonus idea, not a required part of Lab 2.


In [ ]:
bonus_notes = [
    "Case note: IMG_2044.jpg is the only image attached to the 'black SUV for sale' draft.",
    "Case note: draft-9001 was saved but not published live.",
    "Case note: IMG_2051.jpg shows a black SUV, but the detector did not find a roof rack.",
]


@tool
def search_case_notes(query: str, top_k: int):
    """
    Search short case notes with simple keyword matching.

    Args:
        query (str): Query terms to match against the notes.
        top_k (int): Maximum number of notes to return.
    """
    def clean_token(text: str) -> str:
        return "".join(ch for ch in text.lower() if ch.isalnum())

    query_tokens = {clean_token(token) for token in query.split()}
    query_tokens.discard("")

    scored = []
    for note in bonus_notes:
        note_tokens = {clean_token(token) for token in note.split()}
        note_tokens.discard("")
        overlap = sorted(query_tokens & note_tokens)
        if overlap:
            scored.append({"note": note, "matched_terms": overlap, "score": len(overlap)})
    scored.sort(key=lambda item: item["score"], reverse=True)
    return json.dumps(scored[:top_k], indent=2)

bonus_tool = search_case_notes
{
    "tool_signature": json.loads(bonus_tool.fn_signature),
    "sample_result": json.loads(bonus_tool.run(query="listing draft image", top_k=2)),
}


## Manual vs. Packaged Tool Use

`Part 1` made the tool names, arguments, and raw outputs explicit. `Part 2` handed the same tools to `ToolAgent`, which selected calls and assembled the report automatically. The optional extension shows how the same pattern can later expand to retrieval-style helpers without making vector databases part of the required lab.
